In [1]:
# ---
# This script will load the coastal circle-avg. SST data, put into a matrix, and generate MHW days. (with the 11 DAY SMOOTHING OF SST CLIMATOLOGY!)
## Hobday's paper and other ref's aren't super clear on how this is done, but I'm going to assume that you take 90th percentile for every day of year,
## and then smooth those 90th percentiles with the 11 day rolling window. 

# Noel Siegert, 2/19/25 
# ---

In [2]:
# imports
import os
import xarray as xr
import numpy as np
import netCDF4 
import glob
import pandas as pd
from datetime import datetime
from scipy import stats

In [3]:
# interactive plotting stuff 
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import colors
import matplotlib.cm as cm
import matplotlib.gridspec as gridspec
from matplotlib import rcParams
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

#import matplotlib.dates as mdates
%matplotlib inline
plt.rcParams['figure.figsize'] = 12, 6
#%config InlineBackend.figure_format = 'retina'

import cartopy
import cartopy.crs as ccrs
from cartopy.util import add_cyclic_point

/opt/sw/anaconda3/2023.09/envs/pangeo23/lib/python3.11/site-packages/pyproj/__init__.py:89: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()


In [4]:
script = 'code/analysis/ID_MHW_expanded_2_19.ipynb'

In [5]:
clim_pd = ['1990-01-01', '2023-12-31']
period = clim_pd # (for now)

In [6]:
# dataframe with the stations we are using
df = pd.read_csv('/home/nsiegert/projects/coastal_sst/data/hadisd_stations_using_Expanded.csv')
df = df.drop(['Unnamed: 0'], axis=1)
df

,STAID,STANAME,LAT,LON,ELEV,START,END,DIST2COAST,PCTREPORTING,max_pctmissing_month
0,010014-99999,SORSTOKKEN,59.792,5.341,48.8,1986-11-20,2024-02-02,1.377415,82.28,26.021505
1,011020-99999,SKLINNA FYR,65.200,11.000,16.0,1975-02-28,2024-02-02,23.468849,92.73,16.646989
2,011120-99999,BRONNOY,65.461,12.218,7.6,1973-01-01,2024-02-02,0.976040,96.37,12.514758
3,011160-99999,STOKKA,65.950,12.467,17.1,1973-04-01,2024-02-02,1.240810,95.79,15.702479
4,011210-99999,SOLVAER III,66.367,12.617,10.0,1973-01-01,2024-02-02,10.035263,87.32,17.222222
...,...,...,...,...,...,...,...,...,...,...
1469,987520-99999,BUTUAN,8.950,125.483,46.0,1981-01-01,2024-02-02,4.531335,96.62,6.666667
1470,987530-99999,FRANCISCO BANGOY INTL,7.126,125.646,29.3,1974-06-01,2024-02-02,1.183564,99.53,3.111111
1471,987550-99999,HINATUAN,8.367,126.333,3.0,1949-10-01,2023-12-07,1.553108,96.70,6.021505
1472,988360-99999,ZAMBOANGA INTL,6.922,122.060,10.1,1945-03-12,2024-02-02,2.489244,99.64,2.333333


In [7]:
ds = xr.open_dataset('/dx02/data/nsiegert/oisst_station_cirleavg/ALLSTATIONS.1.5deg.daily.sst.1.9.2025.nc')
sstda = ds.sst
sstda

<xarray.DataArray 'sst' (staid: 1474, time: 12783)> Size: 151MB
[18842142 values with dtype=float64]
Coordinates:
  * staid    (staid) <U12 71kB '010014-99999' '011020-99999' ... '999999-12946'
  * time     (time) datetime64[ns] 102kB 1989-01-01T12:00:00 ... 2023-12-31T1...
Attributes:
    long_name:  Daily sea surface temperature
    units:      Celsius
    valid_min:  -300
    valid_max:  4500
    script:     script
    timestamp:  2025-01-09 13:16:05
    desc.:      "coastal SST" generated by taking lat-weighted avg. of all ss...

In [8]:
## ID days with SST > 90th pctile (using 11day rolling window) ##

# gen 90th percentile SST for every day of year, every station
da_90pctiles = sstda.sel(time=slice(clim_pd[0], clim_pd[1])).groupby("time.dayofyear").quantile(0.9)

# make an array where I attach the first 10 days of year to the end (so rolling window doesn't cut off)
ext_90pct_arr = np.zeros(shape=(len(df), 377))
ext_90pct_arr[:, :366] = da_90pctiles.data # fill normal days
ext_90pct_arr[:, 366:] = da_90pctiles.data[:, :11] # extension days
ext_90pct_arr_da = xr.DataArray(ext_90pct_arr, dims=['staid','daynum'], coords={'staid':sstda.staid, 'daynum':np.array(range(377))}) # put into da

# take 11 day rolling window of 90th percentiles
ext_90pct_arr_da_roll11 = ext_90pct_arr_da.rolling(daynum=11, center=True).mean()

# now, select only days 0-366, and fill in the first 5 days that didn't get data
roll11sel = ext_90pct_arr_da_roll11.data[:, :366]
roll11sel[:, :5] = ext_90pct_arr_da_roll11.data[:, 366:371]

# put into DA
da_90pctiles_roll11 = xr.DataArray(roll11sel, dims=['staid', 'dayofyear'], coords={'staid':sstda.staid, 'dayofyear':np.array(range(1, 367))}) 

# compare data to the rolling 90th percentiles
da_above_90pctile = sstda.sel(time=slice(period[0], period[1])).groupby("time.dayofyear") > da_90pctiles_roll11

/opt/sw/anaconda3/2023.09/envs/pangeo23/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/opt/sw/anaconda3/2023.09/envs/pangeo23/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/opt/sw/anaconda3/2023.09/envs/pangeo23/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/opt/sw/anaconda3/2023.09/envs/pangeo23/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/opt/sw/anaconda3/2023.09/envs/pangeo23/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/opt/sw/anaconda3/2023.09/envs/pangeo23/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWa

In [9]:
# take rolling sum of days>90
n_consec = 5
da_above_90pctile_roll = da_above_90pctile.rolling(time=n_consec).sum()
da_above_90pctile_roll

<xarray.DataArray (staid: 1474, time: 12418)> Size: 146MB
array([[nan, nan, nan, ...,  0.,  0.,  0.],
       [nan, nan, nan, ...,  0.,  0.,  0.],
       [nan, nan, nan, ...,  0.,  0.,  0.],
       ...,
       [nan, nan, nan, ...,  0.,  1.,  2.],
       [nan, nan, nan, ...,  0.,  0.,  0.],
       [nan, nan, nan, ...,  0.,  0.,  0.]])
Coordinates:
  * staid      (staid) <U12 71kB '010014-99999' ... '999999-12946'
  * time       (time) datetime64[ns] 99kB 1990-01-01T12:00:00 ... 2023-12-31T...
    dayofyear  (time) int64 99kB 1 2 3 4 5 6 7 8 ... 359 360 361 362 363 364 365

In [10]:
# arrays to hold MHW classifications
da_hw_days = np.zeros(shape=da_above_90pctile.shape)
da_hw_onsetdays = np.zeros(shape=da_above_90pctile.shape)

In [11]:
# for each station ID (row), ID the marine heatwaves.
for sta in range(len(ds.staid)):
    
    # select this station's data. 
    da_above_90pctile_roll_dat_this_sta = da_above_90pctile_roll[sta].data
    
    # for every index where the rolling sum of hw days == n_consec:
    ## (ie where the rolling array = 3, we know there have been 3 consecutive 90+ days in a row)
    for ii in np.argwhere((da_above_90pctile_roll_dat_this_sta==n_consec).data):

        # set that date, plus the prior n_consec-1 days = 1 in the heatwave day array
        i = ii.item()
        da_hw_days[sta, i-(n_consec-1):i+1] = 1

     ## ID onset days
    for jj in np.argwhere(da_hw_days[sta]==1):

        j = jj.item()

         # if that day is a HW day and the previous is not, then it's an onset day and marks a unique event.
        if (da_hw_days[sta, j]==1) and (da_hw_days[sta, j-1]==0):
            da_hw_onsetdays[sta, j] = 1
    
    # print b/c impatient
    if sta%100==0:
        print(sta)

0
100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400


In [12]:
# put into dataArrays
da_hw_days_DA = xr.DataArray(da_hw_days.astype(int), dims=['staid', 'time'], coords={'staid':sstda.staid, 'time':da_above_90pctile.time})
da_hw_onsetdays_DA = xr.DataArray(da_hw_onsetdays.astype(int), dims=['staid', 'time'], coords={'staid':sstda.staid, 'time':da_above_90pctile.time})

# add attrs,
da_hw_days_DA.name='MHW'
da_hw_onsetdays_DA.name='MHW_onsets'
    
hw_ds_out = xr.merge([da_hw_days_DA, da_hw_onsetdays_DA])

hw_ds_out.MHW.attrs['desc.'] = 'MHW days defined as 5+ consecutive days where daily SST is greater than that day\'s climatological 90th percentile. (Climatological 90th percentile has a centered, 11-day rolling window applied)'
hw_ds_out.attrs['script'] = script
now = datetime.now()
hw_ds_out.attrs['timestamp'] = now.strftime("%Y-%m-%d %H:%M:%S")
hw_ds_out.attrs['info'] = 'heatwave designations generated from RAW SST and Tmax data (not detrended/anomalized)'

# save.
hw_ds_out.to_netcdf('/dx02/data/nsiegert/coastal_mhw_data/ALLSTATIONS.1.5deg.marineheatwaves_roll11.nc')

In [13]:
hw_ds_out

<xarray.Dataset> Size: 293MB
Dimensions:     (staid: 1474, time: 12418)
Coordinates:
  * staid       (staid) <U12 71kB '010014-99999' ... '999999-12946'
  * time        (time) datetime64[ns] 99kB 1990-01-01T12:00:00 ... 2023-12-31...
Data variables:
    MHW         (staid, time) int64 146MB 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
    MHW_onsets  (staid, time) int64 146MB 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
Attributes:
    script:     code/analysis/ID_MHW_expanded_2_19.ipynb
    timestamp:  2025-02-19 12:10:30
    info:       heatwave designations generated from RAW SST and Tmax data (n...